In [1]:
from validators import validate_ticket_input
from prompts import *
from groq_client import call_groq_model
from output_parser import parse_json
# from output_builder import build_final_output
from save_output import save_output

import json

In [2]:
customer_query={
"customer_name": "Amit",
"customer_type": "Premium",
"ticket_subject": "Charged twice and no response from support",
"ticket_body": "Hi team, I cancelled my premium subscription last month, but I was still charged again this month. I also noticed that  same invoice amount appears twice on my bank statement. I contacted support two times last week but have not received any proper response. This is extremely frustrating. If this is not resolved today, I will escalate this publicly on LinkedIn and also ask our finance team to block future payments. Please refund the incorrect charge immediately. Regards, Amit",
"product_area": "Billing and subscription",
"previous_interaction_history": "Customer says they contacted support two times last week and did not receive a proper response.",
"sla_tier": "Premium",
"response_tone": "Professional and empathetic",
"business_rules": [
"Do not promise refund before verification.",
"Do not confirm cancellation unless verified.",
"Ask for invoice ID or registered account email if required.",
"Escalate premium customer billing issues to billing support."
]

}

In [3]:
is_valid = validate_ticket_input(customer_query)

print(is_valid)

{'customer_name': 'Amit', 'customer_type': 'Premium', 'ticket_subject': 'Charged twice and no response from support', 'ticket_body': 'Hi team, I cancelled my premium subscription last month, but I was still charged again this month. I also noticed that  same invoice amount appears twice on my bank statement. I contacted support two times last week but have not received any proper response. This is extremely frustrating. If this is not resolved today, I will escalate this publicly on LinkedIn and also ask our finance team to block future payments. Please refund the incorrect charge immediately. Regards, Amit', 'product_area': 'Billing and subscription', 'previous_interaction_history': 'Customer says they contacted support two times last week and did not receive a proper response.', 'sla_tier': 'Premium', 'response_tone': 'Professional and empathetic', 'business_rules': ['Do not promise refund before verification.', 'Do not confirm cancellation unless verified.', 'Ask for invoice ID or r

In [4]:
response = call_groq_model(
    ticket_summarization(customer_query)
)

print(response)

```json
{
  "short_summary": "Customer Amit, a Premium user, was charged twice and received no response from support.",
  "customer_problem": "Charged twice for a cancelled subscription and lack of response from support.",
  "business_impact": "Potential public escalation on LinkedIn and blocked future payments if not resolved.",
  "customer_requested_action": "Refund the incorrect charge immediately.",
  "important_context": [
    "Customer cancelled premium subscription last month",
    "Same invoice amount appears twice on bank statement",
    "Previous interactions with support were unfruitful"
  ],
  "missing_information": [
    "Invoice ID or registered account email for verification"
  ]
}
```

This summary follows the specified criteria:

- Removed emotional noise while preserving urgency: The summary focuses on the core issue and its urgency.
- Identified the core issue: The customer was charged twice for a cancelled subscription and received no response from support.
- Identi

In [5]:
summary = parse_json(response)

summary

{'error': 'Invalid JSON returned',
 'raw_response': '{\n  "short_summary": "Customer Amit, a Premium user, was charged twice and received no response from support.",\n  "customer_problem": "Charged twice for a cancelled subscription and lack of response from support.",\n  "business_impact": "Potential public escalation on LinkedIn and blocked future payments if not resolved.",\n  "customer_requested_action": "Refund the incorrect charge immediately.",\n  "important_context": [\n    "Customer cancelled premium subscription last month",\n    "Same invoice amount appears twice on bank statement",\n    "Previous interactions with support were unfruitful"\n  ],\n  "missing_information": [\n    "Invoice ID or registered account email for verification"\n  ]\n}\n\n\nThis summary follows the specified criteria:\n\n- Removed emotional noise while preserving urgency: The summary focuses on the core issue and its urgency.\n- Identified the core issue: The customer was charged twice for a cancelled

In [6]:
response = call_groq_model(
    ticket_classification(customer_query)
)

print(response)

Based on the provided rules and examples, I will implement a solution to classify customer queries into the given categories.

```python
import re
from typing import Dict

def classify_ticket(ticket: Dict) -> Dict:
    """
    Classify a customer query into primary and secondary categories.

    Args:
    ticket (Dict): A dictionary containing customer query information.

    Returns:
    Dict: A dictionary containing the primary category, secondary categories, category reasoning summary, and confidence score.
    """

    # Initialize the output dictionary
    output = {
        "primary_category": "",
        "secondary_categories": [],
        "category_reasoning_summary": "",
        "confidence_score": 0.0
    }

    # Check for billing issues
    if "Billing and subscription" in ticket["product_area"] and ("charged" in ticket["ticket_subject"].lower() or "invoice" in ticket["ticket_subject"].lower()):
        output["primary_category"] = "BILLING_ISSUE"
        output["category_r

In [7]:
ticket_classification = parse_json(response)

ticket_classification

{'error': 'Invalid JSON returned',
 'raw_response': 'Based on the provided rules and examples, I will implement a solution to classify customer queries into the given categories.\n\npython\nimport re\nfrom typing import Dict\n\ndef classify_ticket(ticket: Dict) -> Dict:\n    """\n    Classify a customer query into primary and secondary categories.\n\n    Args:\n    ticket (Dict): A dictionary containing customer query information.\n\n    Returns:\n    Dict: A dictionary containing the primary category, secondary categories, category reasoning summary, and confidence score.\n    """\n\n    # Initialize the output dictionary\n    output = {\n        "primary_category": "",\n        "secondary_categories": [],\n        "category_reasoning_summary": "",\n        "confidence_score": 0.0\n    }\n\n    # Check for billing issues\n    if "Billing and subscription" in ticket["product_area"] and ("charged" in ticket["ticket_subject"].lower() or "invoice" in ticket["ticket_subject"].lower()):\n  

In [8]:
response = call_groq_model(
    sentiment_classification(customer_query)
)

print(response)

**Sentiment Analysis**

Based on the provided customer query, I will classify the sentiment as follows:

**Sentiment:** NEGATIVE
**Emotion Signals:** FRUSTRATED, URGENT
**Sentiment Reasoning Summary:** The customer is frustrated due to repeated follow-ups and lack of response from support. The customer is also urgent about resolving the issue as they threaten to escalate it publicly on LinkedIn and ask their finance team to block future payments.
**Confidence Score:** 0.9 (high confidence)

**Reasoning:**

1. The customer mentions being "charged twice" and "no response from support", indicating a negative experience.
2. The customer expresses frustration by stating "This is extremely frustrating" and threatens to escalate the issue publicly, indicating a sense of urgency.
3. The customer explicitly states that they will "escalate this publicly on LinkedIn" and "ask our finance team to block future payments" if the issue is not resolved, indicating a high level of frustration and urgenc

In [9]:
sentiment_classification = parse_json(response)

sentiment_classification

{'error': 'Invalid JSON returned',
 'raw_response': '**Sentiment Analysis**\n\nBased on the provided customer query, I will classify the sentiment as follows:\n\n**Sentiment:** NEGATIVE\n**Emotion Signals:** FRUSTRATED, URGENT\n**Sentiment Reasoning Summary:** The customer is frustrated due to repeated follow-ups and lack of response from support. The customer is also urgent about resolving the issue as they threaten to escalate it publicly on LinkedIn and ask their finance team to block future payments.\n**Confidence Score:** 0.9 (high confidence)\n\n**Reasoning:**\n\n1. The customer mentions being "charged twice" and "no response from support", indicating a negative experience.\n2. The customer expresses frustration by stating "This is extremely frustrating" and threatens to escalate the issue publicly, indicating a sense of urgency.\n3. The customer explicitly states that they will "escalate this publicly on LinkedIn" and "ask our finance team to block future payments" if the issue 

In [10]:
response = call_groq_model(
    priority_risk_prompt(customer_query)
)

print(response)

{
    "priority": "URGENT",
    "escalation_risk": "CRITICAL",
    "risk_triggers": [
        "Payment impact",
        "Public escalation threat",
        "Repeated failed support attempts"
    ],
    "recommended_sla_action": "Immediate billing review and response to customer.",
    "reasoning_summary": "Billing impact, public escalation threat, and repeated failed support attempts require urgent handling for premium customer."
}


In [11]:
priority_risk_prompt = parse_json(response)

priority_risk_prompt

{'priority': 'URGENT',
 'escalation_risk': 'CRITICAL',
 'risk_triggers': ['Payment impact',
  'Public escalation threat',
  'Repeated failed support attempts'],
 'recommended_sla_action': 'Immediate billing review and response to customer.',
 'reasoning_summary': 'Billing impact, public escalation threat, and repeated failed support attempts require urgent handling for premium customer.'}

In [12]:
response = call_groq_model(
     sensitive_info_detection(customer_query)
)

print(response)

```python
import json

def detect_sensitive_info(ticket):
    """
    Detects sensitive information in a given ticket.

    Args:
    ticket (dict): A dictionary containing ticket information.

    Returns:
    dict: A dictionary containing the result of sensitive information detection.
    """
    sensitive_categories = []
    evidence_summary = ""
    handling_recommendations = []

    # Check for payment information
    if any(keyword in ticket['ticket_body'].lower() for keyword in ['charge', 'refund', 'invoice', 'payment', 'bank', 'card', 'amount']):
        sensitive_categories.append("PAYMENT_INFORMATION")
        evidence_summary += "Payment information detected in the ticket body.\n"
        handling_recommendations.append("Verify the customer's claim and refund the incorrect charge if necessary.")

    # Check for personal information
    if any(keyword in ticket['ticket_body'].lower() for keyword in ['name', 'email', 'phone', 'address']):
        sensitive_categories.append("

In [13]:
sensitive_info_detection = parse_json(response)

sensitive_info_detection

{'error': 'Invalid JSON returned',
 'raw_response': 'python\nimport json\n\ndef detect_sensitive_info(ticket):\n    """\n    Detects sensitive information in a given ticket.\n\n    Args:\n    ticket (dict): A dictionary containing ticket information.\n\n    Returns:\n    dict: A dictionary containing the result of sensitive information detection.\n    """\n    sensitive_categories = []\n    evidence_summary = ""\n    handling_recommendations = []\n\n    # Check for payment information\n    if any(keyword in ticket[\'ticket_body\'].lower() for keyword in [\'charge\', \'refund\', \'invoice\', \'payment\', \'bank\', \'card\', \'amount\']):\n        sensitive_categories.append("PAYMENT_INFORMATION")\n        evidence_summary += "Payment information detected in the ticket body.\\n"\n        handling_recommendations.append("Verify the customer\'s claim and refund the incorrect charge if necessary.")\n\n    # Check for personal information\n    if any(keyword in ticket[\'ticket_body\'].lower(

In [14]:
response = call_groq_model(
   routing_prompt(customer_query)
)

print(response)

```json
{
    "recommended_team": "BILLING_SUPPORT",
    "routing_reason": "Billing dispute involving duplicate charge and non-response from support.",
    "internal_note": "Verify invoice and cancellation history, and follow business rules.",
    "required_follow_up_information": [
        "Invoice ID",
        "Account Email"
    ]
}
```

This recommendation is based on the following:

- The ticket subject and body indicate a billing dispute involving a duplicate charge.
- The customer is a premium customer, which aligns with the business rule to escalate premium customer billing issues to billing support.
- The customer has previously contacted support without receiving a proper response, which may indicate a need for more thorough handling.
- The business rules for this issue type include verifying the invoice and cancellation history, and not promising a refund before verification.


In [15]:
routing_prompt= parse_json(response)

routing_prompt

{'error': 'Invalid JSON returned',
 'raw_response': '{\n    "recommended_team": "BILLING_SUPPORT",\n    "routing_reason": "Billing dispute involving duplicate charge and non-response from support.",\n    "internal_note": "Verify invoice and cancellation history, and follow business rules.",\n    "required_follow_up_information": [\n        "Invoice ID",\n        "Account Email"\n    ]\n}\n\n\nThis recommendation is based on the following:\n\n- The ticket subject and body indicate a billing dispute involving a duplicate charge.\n- The customer is a premium customer, which aligns with the business rule to escalate premium customer billing issues to billing support.\n- The customer has previously contacted support without receiving a proper response, which may indicate a need for more thorough handling.\n- The business rules for this issue type include verifying the invoice and cancellation history, and not promising a refund before verification.'}

In [16]:
response = call_groq_model(
    draft_generator(
        customer_query,
        customer_query["response_tone"],
        customer_query.get(
                    "business_rules",
            []
    )
)
)
print(response)

{
    "draft_response": "Dear Amit,\n\nThank you for reaching out to us about the issue with your premium subscription. We apologize for the inconvenience and frustration caused by the duplicate charge. We understand that you had previously cancelled your subscription and were charged again this month.\n\nTo better assist you, could you please provide the invoice ID or the registered account email associated with your subscription? This will help us to investigate the matter further.\n\nOnce we receive the necessary information, we will look into this issue and respond to you with an update on the next steps. We appreciate your patience and cooperation in this matter.\n\nBest regards, [Your Name]",
    "response_strategy": "Acknowledge the concern, show empathy, summarize the issue, ask for missing information, explain next steps, and maintain a professional tone.",
    "assumptions": [],
    "information_needed_before_sending": ["invoice ID", "registered account email"]
}


In [17]:
draft_generator = parse_json(response)

draft_generator

{'draft_response': 'Dear Amit,\n\nThank you for reaching out to us about the issue with your premium subscription. We apologize for the inconvenience and frustration caused by the duplicate charge. We understand that you had previously cancelled your subscription and were charged again this month.\n\nTo better assist you, could you please provide the invoice ID or the registered account email associated with your subscription? This will help us to investigate the matter further.\n\nOnce we receive the necessary information, we will look into this issue and respond to you with an update on the next steps. We appreciate your patience and cooperation in this matter.\n\nBest regards, [Your Name]',
 'response_strategy': 'Acknowledge the concern, show empathy, summarize the issue, ask for missing information, explain next steps, and maintain a professional tone.',
 'assumptions': [],
 'information_needed_before_sending': ['invoice ID',
  'registered account email']}

In [19]:
final_output = {
    "summary": summary,
    "ticket_classification": ticket_classification,
    "sentiment_classification": sentiment_classification,
    "priority_and_risk": priority_risk_prompt,
    "sensitive_information_check": sensitive_info_detection,
    "routing_recommendation": routing_prompt,
    "draft_customer_response": draft_generator,
}

In [20]:
import json

with open("final_output.json", "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=4, ensure_ascii=False)